# Full Pipeline on Real OmniDocBench Data — SDPA attention build

This notebook is the canonical end-to-end driver for the adaptive-inference research MVP, updated to use PyTorch's built-in **SDPA** attention backend instead of the optional `flash-attn` package.

**Why SDPA?** On A100 / Ampere+ GPUs, PyTorch's SDPA dispatches to Flash Attention 2 kernels internally. We get most of the FA2 speedup without compiling `flash-attn` (which can take 30+ min on Colab and often fails). When `attn_implementation="sdpa"` is honored by InternVL2's remote model code, inference per page drops ~2–3× versus the eager fallback.

**Sections:**
0. Setup (clone, dependency check, SDPA verification)
1. Build the OmniDocBench data fixture
2. Phase 2 — single-pass baseline at low tile budget
3. Phase 4 — adaptive (verifier-gated reparse)
4. Phase 5 — calibration sweep → frozen budgets
5. Save artifacts to Google Drive (run before closing the tab!)
6. Optional — restore artifacts from a previous Drive save
7. Workflow — round-trip `frozen_budgets.json` back to GitHub

## Colab persistence — read once

`/content/` is **wiped** on tab close, idle disconnect (~90 min), runtime restart, or session-time limit (12h free / 24h Pro).

| Category | Where | Survives tab close? |
|---|---|---|
| Source code | GitHub | ✅ if pushed |
| Real data (`data/omnidocbench/`) | gitignored, Colab VM only | ❌ — re-download (~1 min) |
| Run artifacts (`outputs/runs/...`) | gitignored, Colab VM only | ❌ — re-run model (slow) |
| Frozen budgets | checked into git **and** can be Drive-saved | ✅ via either path |
| Drive (`MyDrive/...`) | Google Drive | ✅ |

**Always run Section 5 (Drive save) before closing the tab.**

## Loading this notebook into Colab from GitHub

Once it's pushed to `main`, open it via:

```
https://colab.research.google.com/github/Michaelhamaty/Resarch_dev/blob/main/notebooks/colab_full_pipeline_attn.ipynb
```

Switch to a GPU runtime (Runtime → Change runtime type → GPU, T4 minimum, A100 preferred) before continuing.

## Section 0 — Setup

Clones the repo, installs dependencies, defensively patches `internvl2.py` to use SDPA if the commit hasn't landed on `main` yet, and verifies the GPU + SDPA backend are ready.

In [ ]:
# code cell 1 — clone or fast-forward the repo. Idempotent.
%cd /content
import os
if not os.path.isdir("Research_claude/.git"):
    !git clone https://github.com/Michaelhamaty/Resarch_dev.git Research_claude
else:
    !cd Research_claude && git pull origin main
%cd /content/Research_claude
!git log --oneline -5

In [ ]:
# code cell 2 — install project deps. Colab preinstalls torch + transformers.
# Notably absent: flash-attn. We use PyTorch's SDPA instead (see code cell 3).
!pip install -q huggingface_hub einops timm
!pip install -q -e .

In [ ]:
# code cell 3 — defensive SDPA patch.
# If the SDPA commit has already landed on main this is a no-op.
# If it hasn't (e.g. you opened this notebook before the push), we
# inject attn_implementation="sdpa" into the AutoModel.from_pretrained
# kwargs so this Colab session doesn't run on slow eager attention.
from pathlib import Path
p = Path("src/adaptive_inference/inference/internvl2.py")
text = p.read_text()
if 'attn_implementation="sdpa"' in text:
    print("OK: attn_implementation='sdpa' already present in internvl2.py.")
else:
    old = '                low_cpu_mem_usage=True,\n            )'
    new = '                low_cpu_mem_usage=True,\n                attn_implementation="sdpa",\n            )'
    if old not in text:
        raise RuntimeError(
            "Could not locate AutoModel.from_pretrained kwargs block — "
            "the adapter source has changed shape. Inspect internvl2.py manually."
        )
    p.write_text(text.replace(old, new, 1))
    print("Patched internvl2.py to use attn_implementation='sdpa'.")

In [ ]:
# code cell 4 — verify GPU + SDPA backend.
# If CUDA is False, switch to a GPU runtime before continuing.
import torch, sys
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU       : {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"  Compute   : sm_{cc[0]}{cc[1]}")
    print(f"  bf16 ok   : {torch.cuda.is_bf16_supported()}")
print()
print("SDPA backends:")
print(f"  flash    : {torch.backends.cuda.flash_sdp_enabled()}")
print(f"  mem_eff  : {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"  math     : {torch.backends.cuda.math_sdp_enabled()}")

## Section 1 — Build the OmniDocBench data fixture

Downloads the OmniDocBench English-table pages, filters to pages with at least 8 non-empty `<td>`/`<th>` cells (the `--min-non-empty-cells 8` filter excludes layout-as-table annotation artifacts), and writes the phase-1 manifests + calibration split.

In [ ]:
# code cell 5 — download + filter OmniDocBench English-table pages.
%cd /content/Research_claude
!python scripts/data/build_omnidocbench_fixture.py --limit 50 --min-non-empty-cells 8

In [ ]:
# code cell 6 — build the phase-1 manifests (records.json + splits).
!python scripts/subset_extraction/build_phase1_manifests.py \
    --config configs/dataset/phase1_omnidocbench.yaml

In [ ]:
# code cell 7 — sanity-check the calibration split (table dimensions per page).
import json
from pathlib import Path
RECS = {r["page_id"]: r for r in json.loads(Path("data/omnidocbench/records.json").read_text())}
SPLIT = json.loads(Path("data/splits/omnidocbench/calibration_split.json").read_text())
print(f"records.json total pages: {len(RECS)}")
print(f"calibration_split pages : {len(SPLIT['page_ids'])}")
print()
for pid in SPLIT["page_ids"]:
    r = RECS[pid]
    print(f"  rows={r['row_count']:3d} cols={r['col_count']:2d}  {pid}")

## Section 2 — Phase 2 baseline (single-pass at low tile budget)

Runs the real InternVL2-2B adapter on every calibration page at the configured low budget (4 tiles). Produces `outputs/runs/colab_real_2b_omnidocbench_low_v1/` with per-page sidecars and raw outputs, then scores against the OmniDocBench gold.

In [ ]:
# code cell 8 — Phase 2 single-pass at low tile budget.
# Wipe any stale outputs so we don't confuse a new run with an old one.
!rm -rf outputs/runs/colab_real_2b_omnidocbench_low_v1 outputs/analysis/score_real_v1

!python scripts/main_runs/run_single_pass.py \
    --config configs/runs/colab_real_2b_omnidocbench_low.yaml

In [ ]:
# code cell 9 — score the Phase 2 baseline run.
!python scripts/analysis/score_run.py \
    --run-dir outputs/runs/colab_real_2b_omnidocbench_low_v1 \
    --ground-truth data/omnidocbench/ground_truth.json \
    --output-dir outputs/analysis/score_real_v1

print("=" * 70); print("BASELINE SUMMARY (single-pass low budget)"); print("=" * 70)
!cat outputs/analysis/score_real_v1/summary.md

## Section 3 — Phase 4 adaptive (verifier-gated reparse to high budget)

Same model + dataset as Section 2, but the deterministic structural verifier inspects every low-budget output. Pages with a parse failure are reparsed once at the high budget. Writes `first_pass/`, `reparse/`, and `final/` under the run root; the scorer reads from `final/pages/` automatically.

In [ ]:
# code cell 10 — Phase 4 adaptive run.
!rm -rf outputs/runs/colab_real_2b_omnidocbench_adaptive_v1 outputs/analysis/score_real_adaptive_v1

!python scripts/main_runs/run_adaptive.py \
    --config configs/runs/colab_real_2b_omnidocbench_adaptive.yaml

In [ ]:
# code cell 11 — score the Phase 4 adaptive run.
!python scripts/analysis/score_run.py \
    --run-dir outputs/runs/colab_real_2b_omnidocbench_adaptive_v1 \
    --ground-truth data/omnidocbench/ground_truth.json \
    --output-dir outputs/analysis/score_real_adaptive_v1

print("=" * 70); print("ADAPTIVE SUMMARY"); print("=" * 70)
!cat outputs/analysis/score_real_adaptive_v1/summary.md

## Section 4 — Phase 5 calibration (the step we are currently on)

Sweeps a small grid of adaptive (low, high) pairs + fixed-2B budgets + fixed-8B-stub budgets on the 20-page calibration split, then picks the matched-budget triple and writes `configs/calibration/frozen_budgets.json`. Target adaptive cost is 10.0 tiles/page.

**Expected runtime**: ~10–15 min on A100 with SDPA enabled. Significantly longer on eager attention.

In [ ]:
# code cell 12 — run the Phase 5 sweep.
# Wipe stale sweep state so an earlier partial run doesn't taint the artifact.
!rm -rf outputs/calibration/sweep_omnidocbench outputs/calibration/sweep_omnidocbench_summaries.jsonl

!python scripts/calibration/run_calibration.py \
    --config configs/calibration/phase5_omnidocbench.yaml

In [ ]:
# code cell 13 — inspect the frozen budgets that just got picked.
import json
from pathlib import Path

frozen = json.loads(Path("configs/calibration/frozen_budgets.json").read_text())

print(f"run_id           : {frozen['run_id']}")
print(f"generated_at     : {frozen['generated_at']}")
print()
print("FROZEN BUDGETS")
for name, b in frozen["budgets"].items():
    print(f"  {name:10s}  max_tiles={b['max_tiles']:3d}  "
          f"model={b['model_name']:14s}  adapter={b['adapter_kind']}")
print()
print("SELECTION")
sel = frozen["selection"]
print(f"  adaptive : low={sel['adaptive']['low_max_tiles']} "
      f"high={sel['adaptive']['high_max_tiles']}  "
      f"target={sel['adaptive']['target_cost_tiles']:.2f}  "
      f"measured={sel['adaptive']['measured_cost_tiles']:.2f}")
print(f"  fixed_2b : max_tiles={sel['fixed_2b']['max_tiles']}  "
      f"target={sel['fixed_2b']['target_cost_tiles']:.2f}  "
      f"measured={sel['fixed_2b']['measured_cost_tiles']:.2f}  "
      f"within_tolerance={sel['fixed_2b']['within_tolerance']}")
print(f"  fixed_8b : max_tiles={sel['fixed_8b']['max_tiles']}  "
      f"target={sel['fixed_8b']['target_cost_tiles']:.2f}  "
      f"measured={sel['fixed_8b']['measured_cost_tiles']:.2f}  "
      f"within_tolerance={sel['fixed_8b']['within_tolerance']}")

In [ ]:
# code cell 14 — full sweep table (every (budget, cost, reparse_rate) point measured).
import json
print(f"{'sweep':10s}  {'tiles':14s}  {'cost':>6s}  {'reparse_rate':>12s}")
print("=" * 60)
with open("outputs/calibration/sweep_omnidocbench_summaries.jsonl") as f:
    for line in f:
        d = json.loads(line)
        kind = d.get("sweep_kind", "?")
        if kind == "adaptive":
            tiles = f"low={d['low_max_tiles']} high={d['high_max_tiles']}"
            rep = f"{d.get('reparse_rate', 0.0):.2f}"
        else:
            tiles = f"max={d['max_tiles']}"
            rep = "\u2014"
        print(f"{kind:10s}  {tiles:14s}  {d['cost_tiles']:6.2f}  {rep:>12s}")

## Section 5 — Save artifacts to Google Drive

**Run this before closing the tab.** Copies run artifacts, scorer outputs, frozen budgets, and the (gitignored) dataset files to `MyDrive/research_claude_session/` so a fresh Colab session can resume without re-running Phase 2/4/5.

In [ ]:
# code cell 15 — mount Drive and copy artifacts.
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/research_claude_session")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

TO_SAVE = [
    "outputs/analysis",
    "outputs/calibration",
    "outputs/runs",
    "configs/calibration/frozen_budgets.json",
    "data/omnidocbench/records.json",
    "data/omnidocbench/ground_truth.json",
    "data/splits/omnidocbench",
]
for src in TO_SAVE:
    src_path = Path(src)
    if not src_path.exists():
        print(f"  SKIP {src} (not found on disk)")
        continue
    dst = DRIVE_DIR / src
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
    else:
        shutil.copy2(src_path, dst)
    print(f"  saved {src}")
print()
print(f"Drive contents now at {DRIVE_DIR}:")
!ls -la /content/drive/MyDrive/research_claude_session/

## Section 6 — Recovering from Drive on a fresh Colab session

Optional. Only run this if you want to skip re-running Phase 2/4/5 in a new session — copies the artifacts back from Drive into the freshly-cloned repo.

In [ ]:
# code cell 16 — restore artifacts from Drive (optional).
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/research_claude_session")
REPO = Path("/content/Research_claude")

TO_RESTORE = [
    "outputs/analysis",
    "outputs/calibration",
    "outputs/runs",
    "configs/calibration/frozen_budgets.json",
    "data/omnidocbench/records.json",
    "data/omnidocbench/ground_truth.json",
    "data/splits/omnidocbench",
]
for rel in TO_RESTORE:
    src = DRIVE_DIR / rel
    dst = REPO / rel
    if not src.exists():
        print(f"  SKIP {rel} (not in Drive)")
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f"  restored {rel}")

## Section 7 — Commit `frozen_budgets.json` back to GitHub

Colab cannot push to GitHub directly. Round-trip the regenerated `frozen_budgets.json`:

1. Run Section 5 (Drive save) — the new `frozen_budgets.json` is now in `MyDrive/research_claude_session/configs/calibration/`.
2. On your Mac (Google Drive desktop client or the web download):

    ```bash
    cp ~/Google\ Drive/MyDrive/research_claude_session/configs/calibration/frozen_budgets.json \
       /Users/michaelhamaty/Developer/Research_claude/configs/calibration/frozen_budgets.json

    cd /Users/michaelhamaty/Developer/Research_claude
    git diff configs/calibration/frozen_budgets.json
    git add configs/calibration/frozen_budgets.json
    git commit -m "Refreeze budgets from real Phase 5 calibration on OmniDocBench"
    git push origin main
    ```

3. Once it's on `main`, Phase 6 (multi-system run) can consume it.

## What comes next

- Phase 6: run all five systems with the frozen budgets and collect scored outputs.
- Phase 7: aggregate accuracy + cost into the matched-budget comparison table.
- `project_status_final.md` is the final deliverable, written at the very end with honest numbers.